# Dino Slayer: rebuilding the three tables

How `training_table.csv`, `tower_pairs.csv` and `tower_isolated.csv` are
produced, and which of them this notebook can actually rebuild.

## Read this first

Two of the three rebuild from the repo with plain Python. The fourth file in
the chain, `tower_pairs_los.csv`, **has no generator in this repository**. It
carries the SRTM terrain screen and was produced once outside this code.
Every step below that depends on it reads the committed copy.

So: this notebook reproduces the pipeline. It does not reproduce the terrain
screen. Section 3 says exactly what would be needed to close that gap.

| File | Rows | Written by | Rebuildable here |
|---|---|---|---|
| `ml/training_table.csv` | 1,448 | `export_training_table.py` | yes |
| `ml/tower_pairs.csv` | 10,070 | `tower_scenarios.py` | yes |
| `ml/tower_pairs_los.csv` | 10,070 | **nothing in the repo** | **no** |
| `ml/tower_isolated.csv` | 449 | `tower_los.py` | yes, from the committed screen |

Run order:

```
export_training_table.py   ->  training_table.csv
tower_scenarios.py         ->  tower_pairs.csv
  [SRTM screen, external]  ->  tower_pairs_los.csv
tower_los.py               ->  tower_isolated.csv  (+ tower_los_scenarios.json)
export_isolated.py         ->  web/isolated.json
```


## 0. Setup

Colab already has pandas and numpy, which is everything these scripts import.
`export_training_table.py` uses the standard library only.


In [ ]:
!git clone --depth 1 https://github.com/RextonRZ/dino-slayer.git
%cd dino-slayer
import pandas, numpy
print('pandas', pandas.__version__, '| numpy', numpy.__version__)


Optional: mount Drive if you want the rebuilt tables written somewhere that
survives the runtime being recycled.


In [ ]:
USE_DRIVE = False   # set True to copy outputs to Drive at the end

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUT = '/content/drive/MyDrive/dino-slayer-tables'
    import os
    os.makedirs(DRIVE_OUT, exist_ok=True)
    print('writing to', DRIVE_OUT)


Keep the committed copies aside, so each rebuild can be **checked** against
them rather than merely declared to work.


In [ ]:
import shutil, pathlib
ML = pathlib.Path('dataset/ml')
REF = pathlib.Path('reference')
REF.mkdir(exist_ok=True)
for f in ['training_table.csv', 'tower_pairs.csv', 'tower_pairs_los.csv',
          'tower_isolated.csv']:
    shutil.copy(ML / f, REF / f)
print('kept', len(list(REF.glob('*.csv'))), 'reference files')


## 1. `training_table.csv`, 1,448 rows

```
python dataset/export_training_table.py
```

One row per settlement. Reads `dataset/web/dipi.geojson` and the boundary
file, joining district and division by point in polygon.

**Feature columns.** `pop_2km`, `n_schools_3km`, `n_clinics_3km`, `rwi`,
`elevation_m`, `seasonal_water_px`, `flood_prone`, `place`, plus three terrain
columns derived in this script so the table, the dashboard and the agent all
share one definition:

- `backhaul_km`, straight line distance to the nearest town or city, from OSM
  place types. A stand in for distance to backhaul, not a survey of where
  fibre actually terminates.
- `elev_drop_m`, metres below or above that town. Masts cluster at towns, so a
  large drop means terrain is more likely in the way. A proxy from point
  elevations. No line of sight is calculated here and none is implied.
- `elev_pct_district`, elevation percentile inside its own district, so 'high'
  is judged locally rather than against sea level.

**Two rules that matter.**

1. No column derived from speed tests is present. Otherwise the model would be
   predicting speed from speed. `ml/model_ablations.json` records why each
   column is in or out.
2. Nothing is imputed. Missing stays empty so XGBoost sees `NaN`.

**The `split` column** is what makes it a training table:

| split | rows | meaning |
|---|---|---|
| `train` | 850 | measured: n_tests >= 20 and n_tiles >= 3 |
| `validate` | 264 | low evidence: n_tests >= 5 |
| `check` | 118 | held out |
| `predict` | 216 | no measurement, the model's actual job |

`division` exists so GroupKFold by district works. That is what produces the
honest spatial score (MAE 34.06, R2 0.330) rather than the flattering random
one (25.31, 0.557).


In [ ]:
!python dataset/export_training_table.py


In [ ]:
import pandas as pd
new = pd.read_csv('dataset/ml/training_table.csv')
ref = pd.read_csv('reference/training_table.csv')
print(f'{len(new):,} rows x {len(new.columns)} cols')
print(new.split.value_counts().to_string())
print()
print('identical to the committed copy:', new.equals(ref))


## 2. `tower_pairs.csv`, 10,070 rows

```
python dataset/tower_scenarios.py
```

Reads `training_table.csv` and `dipi.geojson`. Takes the **449 settlements**
the recommender sends to a tower, then enumerates every candidate mast to
settlement pair within 3, 5 and 10 km.

The recommender's thresholds are restated at the top of the script so it picks
exactly the same 449 the dashboard does: `FIBRE_MAX_KM=15`,
`FIBRE_MIN_POP=3000`, `FWA_MAX_KM=40`, `FWA_MIN_POP=500`,
`TERRAIN_DROP_M=-150`.

**A row at this stage is geometry.** Within the radius, nothing more. No
terrain. Candidate mast sites are existing settlement coordinates, a
settlement centre proxy rather than a confirmed buildable site: nobody has
checked land, access or planning at any of them.

That also means full coverage is always reachable, because a settlement always
covers itself. 'Uncovered' is therefore a check on the arithmetic rather than
a finding.

| Column | Meaning |
|---|---|
| `radius_km` | 3, 5 or 10. A pair appears once per radius it falls inside |
| `mast_id`, `mast_lat`, `mast_lon` | the candidate mast site |
| `s_id`, `s_lat`, `s_lon` | the settlement it might serve |
| `dist_km` | haversine, R = 6371.0088 km |
| `chosen_by_distance` | 1 if the distance only greedy cover picked this mast, 1,260 of 10,070 |

Rows per radius: 1,412 at 3 km, 2,596 at 5 km, 6,062 at 10 km. No self pairs:
a mast in the village has nothing to see over, so the diagonal is set in code
rather than stored.


In [ ]:
!python dataset/tower_scenarios.py


In [ ]:
new = pd.read_csv('dataset/ml/tower_pairs.csv')
ref = pd.read_csv('reference/tower_pairs.csv')
print(f'{len(new):,} rows x {len(new.columns)} cols')
print(new.radius_km.value_counts().sort_index().to_string())
print()
print('same pair set as committed:',
      set(zip(new.radius_km, new.mast_id, new.s_id)) ==
      set(zip(ref.radius_km, ref.mast_id, ref.s_id)))


## 3. `tower_pairs_los.csv`: the step this notebook cannot run

**Nothing in the repository writes this file.** It is the same 10,070 pairs
with a terrain screen applied, produced once against SRTM outside this code.

For each pair a terrain profile is sampled along the path and checked twice:

| Test | Rule | Passes |
|---|---|---|
| `los` | the straight ray from a 30 m mast to a 10 m receiver clears the ground everywhere, with a 4/3 earth radius for refraction | 4,211 (41.8%) |
| `fresnel` | stricter: 60% of the first Fresnel zone also clear, at 700 MHz | 2,808 (27.9%) |

`fresnel` implies `los` by construction and the file confirms it: no path
passes the stricter test while failing the looser one. `tower_los.py` asserts
that on load rather than assuming it.

**This is the headline number.** Distance alone assumed 100% of in radius
paths were usable. With terrain applied, 27.9% survive.

### Columns added on top of `tower_pairs.csv`

| Column | Meaning |
|---|---|
| `min_clear_m` | smallest clearance along the path, negative means blocked. Range -604.4 to 21.5 m |
| `los` | passes the line of sight test |
| `fresnel` | passes the 60% first Fresnel test |
| `voids` | DEM holes hit along the path. **0 everywhere**, asserted before use |
| `usable` | **True on all 10,070 rows**: the profile sampled cleanly. It does NOT mean the link works. Use `fresnel` for that |

`usable` is worth flagging to anyone reading the file. The name suggests a
verdict on the link and it is not one.

### The asymmetry that catches people

The screen is **directed, not symmetric**. The mast is 30 m and the receiver
is 10 m, so the ray from A to B is a different profile from B to A. At 3 km,
**100 of the 712 clear paths work one way only**. Any rebuild has to keep both
directions as separate rows.

### To close this gap

A rebuild would need, in Earth Engine or against a local SRTM tile:

1. read `tower_pairs.csv`, all 10,070 directed pairs
2. sample SRTM elevation along each great circle path
3. add 30 m at the mast end and 10 m at the receiver end
4. apply a 4/3 effective earth radius for refraction
5. compute clearance at every sample, take the minimum, write `min_clear_m`
6. `los` = minimum clearance >= 0
7. `fresnel` = clearance >= 60% of the first Fresnel radius at 700 MHz
8. count DEM voids per path

Until that exists the committed file is the record. The cell below confirms it
is present and self consistent.


In [ ]:
los = pd.read_csv('dataset/ml/tower_pairs_los.csv')
pairs = pd.read_csv('dataset/ml/tower_pairs.csv')

print(f'{len(los):,} screened paths')
print('same pair set as tower_pairs.csv:',
      set(zip(pairs.radius_km, pairs.mast_id, pairs.s_id)) ==
      set(zip(los.radius_km, los.mast_id, los.s_id)))
print('no DEM voids:', int(los.voids.sum()) == 0)
print('fresnel never passes where los fails:', bool((los.fresnel <= los.los).all()))
print()
for r in (3, 5, 10):
    g = los[los.radius_km == r]
    print(f'{r:>2} km  {len(g):>5} paths   '
          f'LoS {g.los.mean()*100:5.1f}%   Fresnel {g.fresnel.mean()*100:5.1f}%')
print()
print(f'overall  LoS {los.los.mean()*100:.1f}%   Fresnel {los.fresnel.mean()*100:.1f}%')


## 4. `tower_isolated.csv`, 449 rows

```
python dataset/tower_los.py
```

Re-runs the greedy set cover over the paths that survived the screen. It falls
out with one row per tower settlement recording whether **any other** candidate
site can reach it, at 3, 5 and 10 km.

| Column | 3 km | 5 km | 10 km |
|---|---|---|---|
| `only_self_*` | 191 | 169 | 157 |

### The distinction the file exists for

```
serves == 1          172 at 3 km   a mast the greedy solver happened to place
                                   that ended up covering only itself.
                                   An artefact of the solution.

only_self_reachable  191 at 3 km   NO other candidate site can reach it.
                                   A property of the terrain, true whatever
                                   solver you run.
```

The second is the one a planner can act on, so it is the one published.

In code the difference is an axis. `only_self` is a **column** sum of the
adjacency matrix, not a row sum: `A[mast][settlement]`, and because the screen
is directed the two differ. The row sum asks 'does this site serve anyone but
itself', a question about candidate masts. The column sum asks 'can anything
but itself reach this settlement', which is the one that costs money.

Greedy set cover is an approximation, not a minimum. Set cover is NP-hard and
greedy carries a ln(n) bound, so every mast count here is an **upper bound**.


In [ ]:
!python dataset/tower_los.py


In [ ]:
new = pd.read_csv('dataset/ml/tower_isolated.csv')
ref = pd.read_csv('reference/tower_isolated.csv')
print(f'{len(new)} tower settlements')
for c in ['only_self_3km', 'only_self_5km', 'only_self_10km']:
    print(f'  {c}: {int(new[c].sum())}')
print()
print('identical to the committed copy:', new.equals(ref))


## 5. Publish to the dashboard

`export_isolated.py` turns the CSV into `dataset/web/isolated.json`, which is
what the dashboard fetches. It refuses to write if the radii do not nest, or
if the counts disagree with `towers_los.json`.


In [ ]:
!python dataset/export_isolated.py


## 6. Copy to Drive


In [ ]:
if USE_DRIVE:
    for f in ['training_table.csv', 'tower_pairs.csv', 'tower_pairs_los.csv',
              'tower_isolated.csv']:
        shutil.copy(ML / f, f'{DRIVE_OUT}/{f}')
    shutil.copy('dataset/web/isolated.json', f'{DRIVE_OUT}/isolated.json')
    print('copied to', DRIVE_OUT)
else:
    print('USE_DRIVE is False, nothing copied')


## How far this goes, and where it stops

A terrain screen is **not a propagation model**. It says a path is
geometrically blocked. It does not say a clear path delivers usable signal.
There is no ground cover, no clutter loss, no interference and no capacity
here. **ITU-R P.1812** is the real calculation, and it is named as the one
this project does not run.

SRTM is C-band radar, so over dense forest the returned surface sits partway
up the canopy rather than at ground level. That cuts both ways: the screen
**over-blocks cleared land** whose true ground is lower than the reading, and
**under-blocks tall forest** a real signal would have to pass through. It
narrows the radius uncertainty. It does not close it.

The 30 m mast is sourced, Oughton 2021 p12. The **10 m receiver height is
unsourced**, and it moves the answer.
